<a href="https://colab.research.google.com/github/Theno1gitty/A-Game-Theory-Simulation-of-the-US-Iran-conflict/blob/main/game_theory_sim_US_Iran.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import itertools
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# I created this for reproducibility
random.seed(42)
np.random.seed(42)


def get_us_strategy(current_round, iran_history, us_p_defect_after_iran_coop):
  if current_round == 0:
    return "Cooperate"
  if iran_history[-1] == "Defect":
    return "Defect" if random.random() < 0.75 else "Cooperate"
  return "Defect" if random.random() < us_p_defect_after_iran_coop else "Cooperate"


def get_iran_strategy(
    current_round, us_history, iran_reserves, iran_p_defect_after_us_coop
):
  if iran_reserves < 300:
    return "Cooperate"
  if current_round == 0:
    return "Defect"
  if us_history[-1] == "Defect":
    return "Defect" if random.random() < 0.80 else "Cooperate"
  return (
      "Defect"
      if random.random() < iran_p_defect_after_us_coop
      else "Cooperate"
  )


def run_simulation(
    max_rounds,
    iran_cost_multiplier,
    initial_us_reserves,
    initial_iran_reserves,
    us_p_defect_after_iran_coop,
    iran_p_defect_after_us_coop,
):
  us_reserves = initial_us_reserves
  iran_reserves = initial_iran_reserves

  us_history = []
  iran_history = []

  for r in range(max_rounds):
    us_move = get_us_strategy(r, iran_history, us_p_defect_after_iran_coop)
    iran_move = get_iran_strategy(
        r, us_history, iran_reserves, iran_p_defect_after_us_coop
    )

    us_history.append(us_move)
    iran_history.append(iran_move)

    us_cost = 50
    iran_cost = 30

    if us_move == "Defect" and iran_move == "Defect":
      us_cost += 400
      iran_cost += 200
    elif us_move == "Defect" and iran_move == "Cooperate":
      us_cost += 150
      iran_cost += 300
    elif us_move == "Cooperate" and iran_move == "Defect":
      us_cost += 250 * iran_cost_multiplier
      iran_cost += 50
    else:  # (Cooperate, Cooperate)
      us_cost -= 20
      iran_cost -= 10

    us_reserves -= us_cost
    iran_reserves -= iran_cost

    # Termination evaluations
    if us_move == "Cooperate" and iran_move == "Cooperate":
      return "Settlement"

    if us_reserves <= 0 and iran_reserves <= 0:
      return "Catastrophe"
    elif us_reserves <= 0:
      return "Iran Strategic Advantage"
    elif iran_reserves <= 0:
      return "Allied Strategic Victory"

  return "Stalemate"


# Parameter Grids (Added 1.5 to IRAN_COST_MULTIPLIER)
MAX_ROUNDS = 15
IRAN_COST_MULTIPLIER = [1.5, 2, 3, 5, 7]
INITIAL_US_RESERVES = [5000, 7000, 9000, 11000]
INITIAL_IRAN_RESERVES = [1000, 2000, 3000, 4000]
US_P_DEFECT_AFTER_IRAN_COOP = [0.2, 0.4, 0.6, 0.8]
IRAN_P_DEFECT_AFTER_US_COOP = [0.2, 0.4, 0.6, 0.8]

# Number of Monte Carlo Simulations
NUM_TRIALS = 500


def run_monte_carlo_sweep():
  combinations = list(
      itertools.product(
          IRAN_COST_MULTIPLIER,
          US_P_DEFECT_AFTER_IRAN_COOP,
          IRAN_P_DEFECT_AFTER_US_COOP,
          INITIAL_US_RESERVES,
          INITIAL_IRAN_RESERVES,
      )
  )

  print(f"Initiating Monte Carlo sweep across {len(combinations)} configurations...")
  results = []

  for alpha, us_p, iran_p, r_us, r_iran in combinations:
    outcomes = [run_simulation(MAX_ROUNDS, alpha, r_us, r_iran, us_p, iran_p) for _ in range(NUM_TRIALS)]

    results.append({
        "alpha_cost_multiplier": alpha,
        "beta_defect_ratio": round(iran_p / us_p, 3),
        "gamma_reserves_ratio": round(r_iran / r_us, 3),
        "initial_us_reserves": r_us,
        "initial_iran_reserves": r_iran,
        "us_p_defect": us_p,
        "iran_p_defect": iran_p,
        "p_allied_victory": outcomes.count("Allied Strategic Victory") * 100 / NUM_TRIALS,
        "p_iran_advantage": outcomes.count("Iran Strategic Advantage") / NUM_TRIALS,
        "p_settlement": outcomes.count("Settlement") / NUM_TRIALS,
        "p_catastrophe": outcomes.count("Catastrophe") / NUM_TRIALS,
        "p_stalemate": outcomes.count("Stalemate") / NUM_TRIALS,
    })

    if len(results) % 150 == 0 or len(results) == len(combinations):
      print(f"Progress: [{len(results)}/{len(combinations)}] configurations completed.")

  df = pd.DataFrame(results)
  df.to_csv("monte_carlo_results.csv", index=False)
  return df


sweep_results_df = run_monte_carlo_sweep()
print("\nSimulation complete. Data stored to 'monte_carlo_results.csv'.")
display(sweep_results_df.head(10))

# --- HEATMAP GENERATION ---
print("\nGenerating The Heatmaps...")
sns.set_theme()
fig, axes = plt.subplots(1, 5, figsize=(32, 7.5), sharey=True)

sweep_results_df["gamma_bin"] = pd.cut(
    sweep_results_df["gamma_reserves_ratio"],
    bins=[0.0, 0.1, 0.3, 0.5, 0.8, 1.0],
    labels=["< 0.1", "0.1 - 0.3", "0.3 - 0.5", "0.5 - 0.8", "0.8 - 1.0"],
    include_lowest=True,
)

sweep_results_df["beta_bin"] = pd.cut(
    sweep_results_df["beta_defect_ratio"],
    bins=[0.0, 0.9, 1.1, 4.5],
    labels=["< 1 (US Aggressive)", "= 1 (Symmetric)", "> 1 (Iran Aggressive)"],
    include_lowest=True,
)

for i, alpha in enumerate(IRAN_COST_MULTIPLIER):
  subset = sweep_results_df[sweep_results_df["alpha_cost_multiplier"] == alpha]
  pivot = subset.pivot_table(index="gamma_bin", columns="beta_bin", values="p_allied_victory", observed=False)

  sns.heatmap(
      pivot,
      ax=axes[i],
      cmap="mako",
      annot=pivot.map(lambda v: f"{v:.0f}%"),
      fmt="",
      cbar=(i == 4),
      vmin=0,
      vmax=100,
      linewidths=1,
  )

  axes[i].set_title(f"α = {alpha}")
  axes[i].set_xlabel("Beta Ratio")
  axes[i].tick_params(axis="x", rotation=0, labelsize=10)
  axes[i].tick_params(axis="y", labelsize=10)
  axes[i].set_ylabel("Gamma Ratio" if i == 0 else "")

plt.suptitle("Phase Diagram: Allied Victory Probability Pr(Allied Victory) under Monte Carlo Sweep")
plt.tight_layout()
plt.savefig("allied_victory_heatmap.png")
plt.show()
